# RAG Step 5: Generation and End-to-End Pipeline
Feed the augmented prompt to an LLM. We use a small local model so no API key is needed.

In [ ]:
!pip install -q transformers torch sentence-transformers chromadb

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline

embedder = SentenceTransformer("all-MiniLM-L6-v2")
col = chromadb.PersistentClient(path="./chroma_store").get_collection("rag_demo")
generator = pipeline("text2text-generation", model="google/flan-t5-base")

## The RAG pipeline

In [ ]:
def rag_answer(question, k=3, show_context=False):
    q = embedder.encode([question], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q, n_results=k)
    context = "\n".join(res["documents"][0])
    prompt = (
        "Answer the question using only the context. If unsure, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    answer = generator(prompt, max_new_tokens=80)[0]["generated_text"]
    return (answer, res["documents"][0]) if show_context else answer

print(rag_answer("What is RAG?"))
print(rag_answer("Why does chunk overlap help?"))

## With vs without retrieval

In [ ]:
q = "Which similarity metric is common for normalised vectors?"
print("No RAG  :", generator(q, max_new_tokens=40)[0]["generated_text"])
print("With RAG:", rag_answer(q))

## Swapping in a hosted LLM
Replace the `generator` call with any chat API (OpenAI, Anthropic, etc.). Keep the key in an environment variable, never in the notebook.

In [ ]:
# import os, anthropic
# client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
# msg = client.messages.create(model="<model-name>", max_tokens=300,
#                              messages=[{"role": "user", "content": prompt}])
# print(msg.content[0].text)

## Takeaways
RAG = retrieve + augment + generate. The generator is swappable; the retrieval quality matters most.